# Build Planck CMB Covariance Matrices
Constructs TT, TE, and EE covariance matrices from Planck power spectra using
skew-normal sampling on the asymmetric error bars. Set `FORCE_RECOMPUTE = True`
to regenerate a matrix even if the output file already exists.

In [2]:
import os
import numpy as np
from scipy.stats import skewnorm

# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR   = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "Planck")
INPUT_DIR  = DATA_DIR
OUTPUT_DIR = DATA_DIR   # CSVs saved alongside the input data

NUM_SAMPLES     = 50_000
SEED            = 314100
FORCE_RECOMPUTE = False  # set True to overwrite existing CSV files

In [3]:
def _sample_asymmetric(Dl, err_neg, err_pos, num_samples):
    """Draw samples from a skew-normal fitted to asymmetric error bars."""
    std_dev    = (err_pos + err_neg) / 2
    skew_param = (err_pos - err_neg) / (err_pos + err_neg) * 10
    return skewnorm.rvs(a=skew_param, loc=Dl, scale=std_dev, size=num_samples)


def build_cov_matrix(dl, sdl_neg, sdl_pos, num_samples=NUM_SAMPLES, seed=SEED):
    """Return the sample covariance matrix for a given power spectrum."""
    np.random.seed(seed)  # scipy still uses the global numpy seed
    n       = len(dl)
    samples = np.zeros((num_samples, n))
    for i in range(n):
        samples[:, i] = _sample_asymmetric(dl[i], sdl_neg[i], sdl_pos[i], num_samples)
    return np.cov(samples.T)


def compute_and_save(label, input_file, output_file):
    """Build a covariance matrix and save it, skipping if the file exists."""
    out_path = os.path.join(OUTPUT_DIR, output_file)

    if os.path.exists(out_path) and not FORCE_RECOMPUTE:
        print(f"{label}: already exists at {out_path!r} — skipping.")
        return

    in_path = os.path.join(INPUT_DIR, input_file)
    data    = np.loadtxt(in_path)
    _, dl, sdl_neg, sdl_pos = data[:, 0], data[:, 1], data[:, 2], data[:, 3]

    print(f"{label}: building covariance matrix ({len(dl)} multipoles, {NUM_SAMPLES:,} samples)…")
    cov = build_cov_matrix(dl, sdl_neg, sdl_pos)
    np.savetxt(out_path, cov, delimiter=",")
    print(f"{label}: saved to {out_path!r}")

In [4]:
# ── TT ───────────────────────────────────────────────────────────────────────
compute_and_save(
    label       = "TT",
    input_file  = "COM_PowerSpect_CMB-TT-full_R3.01.txt",
    output_file = "dlstt_cov_matx(mcmc).csv",
)

TT: building covariance matrix (2507 multipoles, 50,000 samples)…
TT: saved to '/Users/leo/Documents/cyi/repos/cmb/SkySimulation/examples/../data/Planck/dlstt_cov_matx(mcmc).csv'


In [5]:
# ── TE ───────────────────────────────────────────────────────────────────────
compute_and_save(
    label       = "TE",
    input_file  = "COM_PowerSpect_CMB-TE-full_R3.01.txt",
    output_file = "dlste_cov_matx(mcmc).csv",
)

TE: building covariance matrix (1995 multipoles, 50,000 samples)…
TE: saved to '/Users/leo/Documents/cyi/repos/cmb/SkySimulation/examples/../data/Planck/dlste_cov_matx(mcmc).csv'


In [6]:
# ── EE ───────────────────────────────────────────────────────────────────────
compute_and_save(
    label       = "EE",
    input_file  = "COM_PowerSpect_CMB-EE-full_R3.01.txt",
    output_file = "dlsee_cov_matx(mcmc).csv",
)

EE: building covariance matrix (1995 multipoles, 50,000 samples)…
EE: saved to '/Users/leo/Documents/cyi/repos/cmb/SkySimulation/examples/../data/Planck/dlsee_cov_matx(mcmc).csv'
